# Templatized training notebook

1. Authenticate with AWS
    - if on local machine 
        - `aws sso login --profile BayesianHealthDevelopers-359300513585`
        - `eval "$(aws configure export-credentials --profile BayesianHealthDevelopers-359300513585 --format env)"`
    - if on SageMaker
        - run local machine setup, copy `AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN`, then run this in SageMaker `export AWS_ACCESS_KEY_ID=XXX AWS_SECRET_ACCESS_KEY=XXX AWS_SESSION_TOKEN=XXX`
2. Run `setup_env.sh`
3. Select `python_311_env` as kernel (unless you specified something different during `setup_env.sh` call)

- Add database configs to local.env

```
{INSTITUTION}_USER=XXXXXXXX
{INSTITUTION}_PASSWORD=XXXXXXXX
{INSTITUTION}_ENGINE=XXXXXXXX
{INSTITUTION}_HOST=XXXXXXXX
{INSTITUTION}_PORT=XXXXXXXX
```

## Imports

In [0]:
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from importlib import reload
import pickle
import time
import os
import polars as pl
import pandas as pd
import numpy as np
import asyncpg
from sklearn.model_selection import train_test_split
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
import matplotlib.pyplot as plt
from xgboost import XGBClassifier, XGBRegressor
import boto3
from io import BytesIO

import train.sepsis.model.mixture_of_experts_training as moe_training
from model.abstract_embedded_model import SepsisMixtureOfExperts

from deterioration.v2.model import DeteriorationModel
from deterioration.v2.deterioration_feature_set import FEATURE_SET
from deterioration.v2.deterioration_constants import SORTED_FEATURES, INTERACTION_TERMS, ABS_DELTA_ORDER, COVARIATE_ORDER
from features.constants.sepsis import BASELINE_FEATURES as SEPSIS_BASELINE_FEATURES
from features.types.feature_set import FeatureSet
from features.types.float_type import FloatFeatureType
from features.types.boolean_type import BooleanFeatureType
from db.data_access import DataAccess

reload(moe_training)
SepsisMixtureOfExpertsTraining = moe_training.SepsisMixtureOfExpertsTraining

model = DeteriorationModel(online_prediction=False)
feature_set = model.feature_set

from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path(os.getcwd()) / "../../../../local-config/local.env", override=True)

import logging
root = logging.getLogger()
root.setLevel(logging.ERROR)

for handler in root.handlers:
    handler.setLevel(logging.ERROR)

### Env setups

In [0]:
utc_ts = datetime(2026, 1, 15, 18, 0, tzinfo=timezone.utc)
tz_str = "America/New_York"

local_dt = utc_ts.astimezone(ZoneInfo(tz_str))
ZoneInfo("America/New_York")

%env ENC_IDENTIFIER_COL_NAMES enc_id
%env ADJUST_FOR_WITHIN_ENC_BASELINE True 
%env USE_NULL_TIMEZONE False

%load_ext autoreload
%autoreload 2

In [0]:
AWS_ACCESS_KEY_ID=
AWS_SECRET_ACCESS_KEY=
AWS_SESSION_TOKEN=

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
)

In [0]:
# Load target enc_ids
def load_target_enc_ids(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(BytesIO(obj["Body"].read()))["enc_id"].tolist()

enc_ids = load_target_enc_ids(
    bucket="data-science-bayesian",
    key="templates/model_training/memorialcare_example_enc_ids.csv",
)

In [0]:


async def extract_pat_enc(db_pool, enc_ids):
    async with db_pool.acquire() as conn:
        rows = await conn.fetch("""
            SELECT * FROM pat_enc where enc_id = ANY($1)
        """, enc_ids)
        return pl.DataFrame([dict(row) for row in rows])

async def extract_cdm_t(db_pool, enc_ids):
    async with db_pool.acquire() as conn:
        rows = await conn.fetch("""
            SELECT * FROM cdm_t where enc_id = ANY($1)
        """, enc_ids)
        return pl.DataFrame([dict(row) for row in rows])

async def extract_cdm_s(db_pool, enc_ids):
    async with db_pool.acquire() as conn:
        rows = await conn.fetch("""
            SELECT * FROM cdm_s where enc_id = ANY($1)
        """, enc_ids)
        return pl.DataFrame([dict(row) for row in rows])

async def extract_enc_care_unit(db_pool, enc_ids):
    async with db_pool.acquire() as conn:
        rows = await conn.fetch("""
            SELECT * FROM enc_care_unit where enc_id = ANY($1)
        """, enc_ids)
        return pl.DataFrame([dict(row) for row in rows])

async def extract_care_units(db_pool, INSTITUTION):
    async with db_pool.acquire() as conn:
        rows = await conn.fetch("""
            SELECT * FROM care_units where institution = $1
        """, INSTITUTION)
        return pl.DataFrame([dict(row) for row in rows])


async def retreive_data(enc_ids: list[int], db_pool, institution):
    cdm_t_df = await extract_cdm_t(db_pool,enc_ids)
    
    cdm_t_df = cdm_t_df.with_columns(
        pl.when((pl.col("value") == ""))
        .then(None)
        .otherwise(pl.col("value"))
        .alias("value")
    )
    
    cdm_s_df = await extract_cdm_s(db_pool, enc_ids)
    ecu_df = await extract_enc_care_unit(db_pool=db_pool, enc_ids=enc_ids)
    prior_encounter_data = await DataAccess.extract_select_fids_from_last_encounter(
        db_pool=db_pool,
        enc_ids=enc_ids,
        fids=SEPSIS_BASELINE_FEATURES.keys(),
    )

    pat_enc_df = await extract_pat_enc(db_pool, enc_ids)
    care_units = await extract_care_units(db_pool, institution)
    return cdm_t_df, cdm_s_df, ecu_df, prior_encounter_data, pat_enc_df, care_units


In [0]:

async def create_pool(ENV):
    return await asyncpg.create_pool(
        host=os.environ.get(f"{ENV}_HOST"), 
        database=os.environ.get(f"{ENV}_NAME"),  
        user=os.environ.get(f"{ENV}_USER"),  
        password=os.environ.get(f"{ENV}_PASSWORD"), 
    )


async def retreive_and_format_data(enc_ids, ENV):
    db_pool = await create_pool(ENV)
    print("pool created")

    cdm_t_df, cdm_s_df, ecu_df, prior_encounter_data, pat_enc_df, care_units_df = await retreive_data(enc_ids, db_pool, ENV)
    await db_pool.close()

    enc_level_of_care_df = ecu_df.join(
        care_units_df.select(
            [
                "care_unit",
                "level_of_care",
            ]
        ),
        on = 'care_unit',
        how = 'left'
    )

    return cdm_t_df, cdm_s_df, ecu_df, prior_encounter_data, pat_enc_df, care_units_df, enc_level_of_care_df

In [0]:
cdm_t_df, cdm_s_df, ecu_df, prior_encounter_data, pat_enc_df, care_units_df, enc_level_of_care_df = await retreive_and_format_data(enc_ids, "MEMORIALCARE")

In [0]:
def get_baseline_cdm(target_enc_ids, cdm_t_df, pat_enc_df, baseline_features):
    """
    Generates a baseline CDM dataframe for a list of target encounters.
    Deduplicates on [enc_id, fid, tsp] keeping the max created_tsp.
    """
    cdm_lazy = cdm_t_df.lazy()
    pat_enc_lazy = pat_enc_df.lazy()

    target_times = (
        cdm_lazy
        .filter(pl.col("enc_id").is_in(target_enc_ids))
        .group_by("enc_id")
        .agg(pl.col("tsp").min().alias("anchor_tsp"))
    )

    target_patients = (
        pat_enc_lazy
        .filter(pl.col("enc_id").is_in(target_enc_ids))
        .select(["enc_id", "pat_id"])
        .rename({"enc_id": "target_enc_id"})
    )

    encounter_map = (
        target_patients
        .join(pat_enc_lazy, on="pat_id")
        .rename({"enc_id": "historical_enc_id"})
    )

    encounter_map = encounter_map.join(
        target_times, 
        left_on="target_enc_id", 
        right_on="enc_id"
    )

    filtered_cdm = cdm_lazy.filter(
        pl.col("fid").is_in(list(baseline_features))
    )

    result = (
        encounter_map
        .join(filtered_cdm, left_on="historical_enc_id", right_on="enc_id")
        .filter(
            pl.col("tsp") < pl.col("anchor_tsp")
        )
        .select([
            pl.col("target_enc_id").alias("enc_id"), 
            "tsp",
            (pl.lit("prior_") + pl.col("fid")).alias("fid"),
            "value"
        ])
        .sort(
            ["enc_id", "fid", "tsp"], 
            descending=[False, False, False]
        )
        .unique(subset=["enc_id", "fid", "tsp"], keep="first")
    )

    return result.collect()

In [0]:
def split_enc_ids(enc_ids, max_enc_ids_per_batch):
    if len(enc_ids) <= max_enc_ids_per_batch:
        return [enc_ids]
    return [enc_ids[i:i+max_enc_ids_per_batch] for i in range(0, len(enc_ids), max_enc_ids_per_batch)]

def calculate_features(
    enc_ids, 
    cdm_t_df, 
    cdm_s_df, 
    pat_enc_df, 
    enc_level_of_care_df, 
    baseline_features, 
    feature_set, 
    folder, 
    file_prefix,
    max_enc_ids_per_batch = 2_000,
    ):
    if not os.path.exists(folder):
        os.makedirs(folder)
    if not os.path.exists(f"{folder}/norm"):
        os.makedirs(f"{folder}/norm")
    if not os.path.exists(f"{folder}/non_norm"):
        os.makedirs(f"{folder}/non_norm")

    enc_chunk_list = split_enc_ids(enc_ids, max_enc_ids_per_batch)
    required_fids, required_prior_fids = feature_set.get_required_fids()

    input_cols = (
        required_fids
        | {f"prior_{fid}" for fid in required_prior_fids}
    )

    pivoted_data_list = []
    norm_feats_list = []
    nonnorm_feats_list = []

    baseline_features = list(SEPSIS_BASELINE_FEATURES.keys())
    enc_chunk = enc_ids

    for i, enc_chunk in enumerate(enc_chunk_list):
        print(f"{datetime.now()}: Pivoting data for chunk {i+1}")
        chunk_filter = pl.col("enc_id").is_in(enc_chunk)
        fid_filter = pl.col("fid").is_in(input_cols)
        df_filter = chunk_filter & fid_filter

        cdm_t_cols = ["enc_id", "tsp", "fid", "value"]
        cdm_s_cols = ["enc_id", "created_tsp", "fid", "value"]

        baselines = get_baseline_cdm(enc_chunk, cdm_t_df, pat_enc_df, baseline_features)

        formatted_data = feature_set.format_data(
            cdm_t_df=cdm_t_df.filter(df_filter).select(cdm_t_cols),
            cdm_s_df=cdm_s_df.filter(df_filter).select(cdm_s_cols),
            prior_enc_df=baselines.filter(df_filter).select(cdm_t_cols),
            level_of_care_df=enc_level_of_care_df.filter(chunk_filter),
            online_prediction=False
        )

        valid_cols = (
            {"enc_id", "tsp"}
            | (input_cols & set(formatted_data.columns))
        )
        formatted_data = formatted_data.select(valid_cols)

        # Need to add UTC timezones until we can turn on USE_NULL_TIMEZONE
        formatted_data = formatted_data.with_columns(
            [
                pl.col(c).dt.replace_time_zone("UTC").alias(c)
                for c in formatted_data.columns if "tsp" in c or c.endswith("time")
            ]
        )
        print(f"{datetime.now()}: Calculating feats for chunk {i+1}")
        norm_feats, nonnorm_feats = feature_set.calculate_features(
            formatted_data, online_prediction=False
        )

        # To avoid keeping each batch of calculated features in memory
        # write them to a file and then read them back lazily
        pivoted_data_file = f"{folder}/{file_prefix}_pivoted-data_batch{i+1}.feather"
        norm_feats_file = f"{folder}/norm/{file_prefix}_feats-normalized_batch{i+1}.feather"
        nonnorm_feats_file = f"{folder}/non_norm/{file_prefix}_feats_batch{i+1}.feather"

        formatted_data.write_ipc(pivoted_data_file, compression="lz4")
        norm_feats.write_ipc(norm_feats_file, compression="lz4")
        nonnorm_feats.write_ipc(nonnorm_feats_file, compression="lz4")

        pivoted_data_list.append(formatted_data)
        norm_feats_list.append(norm_feats)
        nonnorm_feats_list.append(nonnorm_feats)
    print(f"{datetime.now()}: Completed feature calcuation")

    norm_feats_all = pl.concat(norm_feats_list, how="diagonal")
    norm_feats_all.write_ipc(f"{folder}/{file_prefix}_feats-normalized.feather", compression="lz4")

    nonnorm_feats_all = pl.concat(nonnorm_feats_list, how="diagonal")
    nonnorm_feats_all.write_ipc(f"{folder}/{file_prefix}_feats.feather", compression="lz4")
    print(f"{datetime.now()}: Features written to files")

    return norm_feats_all, nonnorm_feats_all




In [0]:
folder = "deterioration_features"
file_prefix = "deterioration"

norm_feats, nonnorm_feats = calculate_features(
    enc_ids, 
    cdm_t_df, 
    cdm_s_df, 
    pat_enc_df, 
    enc_level_of_care_df, 
    SEPSIS_BASELINE_FEATURES,   
    feature_set, 
    folder, 
    file_prefix,
)

# Create labels

In [0]:
def get_label_with_event_tsp(cdm_t_df, cdm_s_df, ecu_df):

    priority_cols = ['care_unit', 'adt_name', 'care_unit_ref']
    stacked_parts = []

    for col_name in priority_cols:
        part = (
            care_units_df
            .lazy().select([
                pl.col(col_name).alias("join_key"),
                pl.lit(col_name).alias("match_source"),
                pl.all()
            ])

            .filter(pl.col("join_key").is_not_null())
        )
        stacked_parts.append(part)

    lookup_df = (
        pl.concat(stacked_parts)
        .unique(subset=["join_key"], keep="first")
        .collect()
    )

    result_df = ecu_df.join(
        lookup_df,
        left_on="care_unit",
        right_on="join_key",
        how="left"
    )

    columns_to_fill = [col for col in lookup_df.columns if col != "join_key"]

    loc = (
        result_df
        .join(
            lookup_df,
            left_on="unmapped_hl7_care_unit",
            right_on="join_key",
            how="left",
            suffix="_fallback"
        )
        .with_columns([
            pl.coalesce([
                pl.col(c), 
                pl.col(f"{c}_fallback")
            ]).alias(c) 
            for c in columns_to_fill
        ])
        .drop([f"{c}_fallback" for c in columns_to_fill])
    )

    enc_id_cols = ['enc_id']
    cdm_t_discharge = cdm_t_df.filter(pl.col("fid") == "discharge")

    cdm_t_deaths = cdm_t_discharge.filter(
        pl.col("value").str.contains("Expired")
    )
    cdm_s_service_type_df = cdm_s_df.filter(pl.col("fid") == "service_type").with_columns(pl.col("value").alias("service_type")).select(["enc_id", "service_type"])
    cdm_t_deaths = cdm_t_deaths.join(
        cdm_s_service_type_df,
        on="enc_id",
        how="left",
    )

    cdm_t_deaths = cdm_t_deaths.sort("tsp")
    loc = loc.sort("enter_time")

    death_tsps_with_ecu = cdm_t_deaths.join_asof(
        loc,
        left_on="tsp",
        right_on="enter_time",
        by=enc_id_cols,
        strategy="backward"
    )

    death_tsps_general_stepdown_obs = death_tsps_with_ecu.filter(
        pl.col("level_of_care").is_in(["observation", "general", "stepdown"])
    )

    death_tsps_general_stepdown_obs_nonhospice = death_tsps_general_stepdown_obs.filter(
        ~pl.col("service_type").str.contains("Hospice|Palliative|Comfort")
    )

    death_tsps_nonhospice = death_tsps_with_ecu.filter(
        ~pl.col("service_type").str.contains("Hospice|Palliative|Comfort")
    )

    discharge_subset = (
        cdm_t_discharge
        .select(enc_id_cols + ["tsp", "value"])
        .rename({"tsp": "discharge_tsp", "value": "dc_dispo"})
        .sort("discharge_tsp")
    )

    ecu_with_discharge_tsps = loc.join_asof(
        discharge_subset,
        left_on="enter_time",
        right_on="discharge_tsp",
        by=enc_id_cols,
        strategy="forward",
    ).with_columns(
        pl.col("dc_dispo").str.contains("Expired").fill_null(False).alias("mortality")
    )


    all_icu_entries = (
        loc.sort("enter_time")
        .with_columns([
            pl.col("care_unit").shift(1).over("enc_id").alias("care_unit_prev"),
            pl.col("level_of_care").shift(1).over("enc_id").alias("level_of_care_prev"),
            pl.col("enter_time").shift(1).over("enc_id").alias("enter_time_prev"),

            pl.col("care_unit").shift(-1).over("enc_id").alias("care_unit_next"),
            pl.col("level_of_care").shift(-1).over("enc_id").alias("level_of_care_next"),
            pl.col("leave_time").shift(-1).over("enc_id").alias("leave_time_next"),
        ])
        .filter(pl.col("level_of_care") == "icu")
    )


    unplanned_icu_xfers = all_icu_entries.filter(
        pl.col("level_of_care_prev").is_in(["general", "stepdown", "observation"])
    )


    unplanned_icu_xfers_ed_ok = all_icu_entries.filter(
        pl.col("level_of_care_prev").is_in(["general", "stepdown", "observation", "emergency"])
    )


    unplanned_icu_xfers_min24h = all_icu_entries.filter(
        pl.col("level_of_care_prev").is_in(["general", "stepdown", "observation"]) &
        (
            (
                (pl.col("leave_time").fill_null(pl.datetime(2125, 1, 1).dt.replace_time_zone("UTC")) - pl.col("enter_time")) 
                >= pl.duration(hours=24)
            ) |
            (pl.col("care_unit_next") == "discharged") |
            (pl.col("level_of_care_next") == "icu") |
            (pl.col("level_of_care_next") == "surgery") |
            (pl.col("level_of_care_next") == "procedure")
        )
    )

    any_icu_xfer_24h = all_icu_entries.filter(
        (
            (pl.col("leave_time").fill_null(pl.datetime(2125, 1, 1).dt.replace_time_zone("UTC")) - pl.col("enter_time")) 
            >= pl.duration(hours=24)
        ) |
        (pl.col("care_unit_next") == "discharged") |
        (pl.col("level_of_care_next") == "icu")
    )


    def prepare_concat(df_icu, df_death, icu_cols, death_cols):
        p1 = df_icu.select(icu_cols).rename({
            "enter_time": "event_tsp", 
            "level_of_care": "outcome"
        })
        
        p2 = df_death.select(death_cols).rename({
            "tsp": "event_tsp", 
            "fid": "outcome"
        })
        
        return (
            pl.concat([p1, p2])
            .with_columns(
                pl.col("outcome").replace("discharge", "mortality")
            )
            .sort("event_tsp")
        )

    # 1. Unplanned Min 24h + Deaths
    unplanned_icu_xfers_min24h_and_death_tsps = prepare_concat(
        df_icu=unplanned_icu_xfers_min24h,
        df_death=death_tsps_general_stepdown_obs,
        icu_cols=enc_id_cols + ["enter_time", "level_of_care"],
        death_cols=enc_id_cols + ["tsp", "fid"]
    )

    # 2. Unplanned Min 24h + Deaths (Non-Hospice)
    unplanned_icu_xfers_min24h_and_death_tsps_nonhospice = prepare_concat(
        df_icu=unplanned_icu_xfers_min24h,
        df_death=death_tsps_general_stepdown_obs_nonhospice,
        icu_cols=enc_id_cols + ["enter_time", "level_of_care"],
        death_cols=enc_id_cols + ["tsp", "fid"]
    )

    # 3. Unplanned ED OK + Deaths (Non-Hospice)
    unplanned_icu_xfers_ed_ok_and_death_tsps_nonhospice = prepare_concat(
        df_icu=unplanned_icu_xfers_ed_ok,
        df_death=death_tsps_nonhospice,
        icu_cols=enc_id_cols + ["enter_time", "level_of_care"],
        death_cols=enc_id_cols + ["tsp", "fid"]
    )

    return unplanned_icu_xfers_min24h_and_death_tsps, unplanned_icu_xfers_min24h_and_death_tsps_nonhospice, unplanned_icu_xfers_ed_ok_and_death_tsps_nonhospice, loc

In [0]:
unplanned_icu_xfers_min24h_and_death_tsps, unplanned_icu_xfers_min24h_and_death_tsps_nonhospice, unplanned_icu_xfers_ed_ok_and_death_tsps_nonhospice, loc = get_label_with_event_tsp(
    cdm_t_df, cdm_s_df, ecu_df
)


In [0]:
def label_and_get_level_of_care(norm_feats, event_tsp_df, loc, enc_id_cols):
    norm_feats = norm_feats.join_asof(
        event_tsp_df,
        left_on="tsp",
        right_on="event_tsp",
        by=enc_id_cols,
        strategy="forward",
        tolerance=timedelta(hours=24)
    )

    norm_feats = norm_feats.join_asof(
        other = loc,
        left_on = 'tsp',
        right_on = 'enter_time',
        by = 'enc_id',
        strategy = 'backward',
    )

    norm_feats = norm_feats.with_columns(
        pl.when(pl.col("outcome").is_in(["mortality", "icu"]))
        .then(1)
        .otherwise(0)
        .alias("label")
    )
    return norm_feats

def filter_by_level_of_care(norm_feats):
    return norm_feats.filter(pl.col('level_of_care').is_in(["general", "observation", "stepdown"]))

norm_feats = label_and_get_level_of_care(
    norm_feats,
    unplanned_icu_xfers_min24h_and_death_tsps_nonhospice,
    loc,
    'enc_id'
    )

norm_feats = filter_by_level_of_care(norm_feats)

In [0]:
def calculate_sample_weights(
    df: pd.DataFrame,
    id_col: str = 'enc_id',
    tsp_col: str = 'tsp', 
    event_tsp_col: str = 'event_tsp',
    washout_hours: int = 48,
    pos_decay_rate: float = 1e-3,
    neg_decay_rate: float = 5e-4,
    min_weight_floor: float = 0.016
) -> pd.DataFrame:

    df['next_event_tsp'] = df.groupby(id_col)[event_tsp_col].bfill()
    df['last_tsp'] = df.groupby(id_col)[tsp_col].transform('max')
    df['t2e_mins'] = (df['next_event_tsp'] - df[tsp_col]).dt.total_seconds() / 60

    is_positive_window = df['next_event_tsp'].notna()
    df['raw_weight'] = 0.0
    if is_positive_window.any():
        pos_t2e = df.loc[is_positive_window, 't2e_mins']
        pos_weights = np.exp(-pos_decay_rate * np.maximum(pos_t2e, -120))
        df.loc[is_positive_window, 'raw_weight'] = np.maximum(pos_weights, min_weight_floor)

    if (~is_positive_window).any():
        neg_t2end = (df.loc[~is_positive_window, 'last_tsp'] - df.loc[~is_positive_window, tsp_col]).dt.total_seconds() / 60
        neg_weights = np.exp(-neg_decay_rate * np.maximum(neg_t2end, -120))
        df.loc[~is_positive_window, 'raw_weight'] = np.maximum(neg_weights, min_weight_floor)
    
    events_only = df[[id_col, event_tsp_col]].dropna().drop_duplicates()
    events_only['prev_event_tsp'] = events_only.groupby(id_col)[event_tsp_col].shift(1)
    df = df.merge(
        events_only[[id_col, event_tsp_col, 'prev_event_tsp']],
        on=[id_col, event_tsp_col],
        how='left'
    )
    df['t_after_e_mins'] = (df[tsp_col] - df['prev_event_tsp']).dt.total_seconds() / 60
    df['t_after_e_mins'] = df['t_after_e_mins'].fillna(np.inf)

    washout_mask = np.where(df['t_after_e_mins'] < washout_hours * 60, 0, 1)
    df['sample_weight'] = df['raw_weight'] * washout_mask
    return df


In [0]:
def get_sample_weight_and_split(norm_feats, SORTED_FEATURES):
    norm_feats_pandas = calculate_sample_weights(norm_feats.to_pandas())
    visit_df = (
        norm_feats_pandas.groupby("enc_id")["label"]
        .max()
        .reset_index()
    )

    train_visits, val_visits = train_test_split(
        visit_df,
        test_size=0.05,
        stratify=visit_df["label"],
        random_state=42
    )

    train_ids = set(train_visits["enc_id"])
    val_ids = set(val_visits["enc_id"])

    train_df = norm_feats_pandas[norm_feats_pandas["enc_id"].isin(train_ids)]
    val_df = norm_feats_pandas[norm_feats_pandas["enc_id"].isin(val_ids)]

    train_df = train_df[SORTED_FEATURES + ['label', 'sample_weight']]
    val_df = val_df[SORTED_FEATURES + ['label', 'sample_weight']]
    return train_df, val_df
train_df, val_df = get_sample_weight_and_split(norm_feats, SORTED_FEATURES)

In [0]:
# """
# Creating feature set for training (converts from new model features to old model features for MoE training)
# """

def create_feature_set(FEATURE_SET, SORTED_FEATURES):
    det_sort_features = set(SORTED_FEATURES)

    features = dict()
    for feature in FEATURE_SET:
        if feature.name in det_sort_features:
            if "float" in str(feature.feature_type.__class__):
                f = FloatFeatureType(
                    default_value=feature.feature_type.default_value,
                    code_missing=False
                )
                f.variable_name_ = feature.name
                features[feature.name] = f
            elif "bool" in str(feature.feature_type.__class__):
                f = BooleanFeatureType(
                    default_value=feature.feature_type.default_value,
                    code_missing=False
                )
                f.variable_name_ = feature.name
                features[feature.name] = f
            else:
                print(feature.name, str(feature.feature_type.__class__))

    sort_features_tuples = [(feat_name, features[feat_name]) for feat_name in SORTED_FEATURES]


    """
    Creating feature set for covariates
    """
    det_covariate_features = set(COVARIATE_ORDER)

    uncoded_values = set(
        ['creatinine_orgdf',
        'bilirubin_orgdf',
        'platelets_orgdf',
        'vent_orgdf',
        'gcs_orgdf',
        'inr_orgdf',
        'hypotension_orgdf',
        'vasopressors_orgdf',
        'lactate_orgdf']
    )

    covariates = dict()
    for feature in FEATURE_SET:
        if feature.name in det_covariate_features:
            if feature.name in uncoded_values:
                covariates[feature.name] = BooleanFeatureType(
                    default_value=False,
                    code_missing=False
                )
            elif "float" in str(feature.feature_type.__class__):
                f = FloatFeatureType(
                    default_value=feature.feature_type.default_value,
                    code_missing=False
                )
                f.variable_name_ = feature.name
                covariates[feature.name] = f
            elif "bool" in str(feature.feature_type.__class__):
                f = BooleanFeatureType(
                    default_value=feature.feature_type.default_value,
                    code_missing=False
                )
                f.variable_name_ = feature.name
                covariates[feature.name] = f
            else:
                print(feature.name, str(feature.feature_type.__class__))

    sort_covariate_tuples = [(feat_name, covariates[feat_name]) for feat_name in COVARIATE_ORDER]


    """
    Creating feature set for det_abs_delta_features
    """
    det_abs_delta_features = set(ABS_DELTA_ORDER)

    abs_deltas = dict()
    for feature in FEATURE_SET:
        if feature.name in det_abs_delta_features:
            if "float" in str(feature.feature_type.__class__):
                f = FloatFeatureType(
                    default_value=feature.feature_type.default_value,
                    code_missing=False
                )
                f.variable_name_ = feature.name
                abs_deltas[feature.name] = f
            elif "bool" in str(feature.feature_type.__class__):
                f = BooleanFeatureType(
                    default_value=feature.feature_type.default_value,
                    code_missing=False
                )
                f.variable_name_ = feature.name
                abs_deltas[feature.name] = f
            else:
                print(feature.name, str(feature.feature_type.__class__))

    sort_abs_deltas_tuples = [(feat_name, abs_deltas[feat_name]) for feat_name in ABS_DELTA_ORDER]

    return sort_features_tuples, sort_covariate_tuples, sort_abs_deltas_tuples
sort_features_tuples, sort_covariate_tuples, sort_abs_deltas_tuples = create_feature_set(FEATURE_SET, SORTED_FEATURES)

# Train with MoE

In [0]:
def train_moe_model(training_set, sort_features_tuples, sort_covariate_tuples, sort_abs_deltas_tuples, SORTED_FEATURES):
    train_feature_set = FeatureSet(sort_features_tuples)
    train_covariates = FeatureSet(sort_covariate_tuples)
    train_abs_delta_feats = FeatureSet(sort_abs_deltas_tuples)
    train_interaction_terms = INTERACTION_TERMS

    num_clusters = [10]
    alpha_regs = [10.0]
    omega_regs = [10.0]
    keep_prob = 1.0
    num_hidden = None

    max_iters = 500
    rand_seed = 10001

    dropout_prob = 1.0 - keep_prob
    for n_c in num_clusters:
        for alpha_reg in alpha_regs:
            for omega_reg in omega_regs:
                key = (dropout_prob, num_clusters, alpha_reg, omega_reg, rand_seed, 
                    'M3-exp_clip - no_regularization',
                    'M3-exp_clip - no_regularization')
                np.random.seed(rand_seed)
                begin_time = time.time()
                print(f"Fitting model with {n_c} clusters, alpha_reg={alpha_reg}, omega_reg={omega_reg} in {max_iters} epochs")
                clf = SepsisMixtureOfExpertsTraining(
                    features=train_feature_set,
                    covariates=train_covariates,
                    abs_delta_feats=train_abs_delta_feats,
                    num_clusters=n_c,
                    alpha_reg=alpha_reg,
                    omega_reg=omega_reg,
                    interaction_terms=train_interaction_terms,
                    keep_prob=keep_prob,
                    num_hidden=num_hidden,
                )
                result = clf.optimize(
                    training_set[SORTED_FEATURES],
                    training_set['label'],
                    sample_weights=training_set['sample_weight'].values,
                    theta0=clf._flatten(),
                    opt_method='adam',
                    max_iters=max_iters,
                    final_bfgs=False,
                    full_batch=True,
                    dropout_prob=dropout_prob,
                )
    description = "Basic model trained using templatized_training_notebook.ipynb"
    name = 'templatized_training_notebook_model'
    new_md = dict()
    new_md["params"] = clf.params
    new_md["model"] = clf.feature_set
    new_md["features"] = clf.feature_set.feature_set
    new_md["covariates"] = clf.covariates
    new_md["abs_delta_feats"] = clf.abs_delta_feats
    new_md["num_clusters"] = 10.0
    new_md["alpha_reg"] = 10.0
    new_md["omega_reg"] = 10.0
    new_md["keep_prob"] = 1
    new_md["num_hidden"] = None
    new_md["interaction_terms"] = INTERACTION_TERMS
    new_md["num_cov"] = clf.covariates.n_columns
    new_md["num_feat"] = clf.feature_set.n_columns
    new_md["num_abs_delta_feat"]=clf.abs_delta_feats.n_columns
    new_md["feature_set_transformations"]=clf.feature_set.transform
    new_md["description"] = description
    new_md["name"] = name
    return new_md

new_md = train_moe_model(train_df, sort_features_tuples, sort_covariate_tuples, sort_abs_deltas_tuples, SORTED_FEATURES)

eval_model = SepsisMixtureOfExperts(
    feature_set_transformations=new_md['model'].transform,
    num_cov=new_md['num_cov'],
    num_feat=new_md['num_feat'],
    num_abs_delta_feat=new_md['num_abs_delta_feat'],
    num_clusters=new_md['num_clusters'],
    alpha_reg=new_md['alpha_reg'],
    omega_reg=new_md['omega_reg'],
    interaction_terms=new_md['interaction_terms'],
    keep_prob=new_md['keep_prob'],
    num_hidden=new_md['num_hidden'],
    params=new_md['params'],
    feature_set=new_md['model'],
    covariates=new_md['covariates'],
    abs_delta_feats=new_md['abs_delta_feats']
)

results_filename_clf_typical_save_format = "MoE_templatized_training_notebook_model.pkl"

buffer = BytesIO()
pickle.dump(new_md, buffer)
buffer.seek(0)
s3.put_object(
    Bucket="data-science-bayesian",
    Key="templates/model_training/MoE_templatized_training_notebook_model.pkl",
    Body=buffer.getvalue(),
)

In [0]:
def predict_and_plot_moe(val_df, SORTED_FEATURES, eval_model):
    val_df['prediction'] = eval_model.predict(
        val_df[SORTED_FEATURES]
    )

    fig, ax = plt.subplots(1,1, figsize = (6,6))
    RocCurveDisplay.from_predictions(val_df['label'], val_df['prediction'], color = 'blue', name = 'example moe model', ax = ax)
    fig.show()
predict_and_plot_moe(val_df, SORTED_FEATURES, eval_model)

# Train XGBoost

In [0]:
def train_xgb_model(training_set, SORTED_FEATURES, label_col = 'label', sample_weight_col = 'sample_weight'):
    clf = XGBClassifier()
    clf.fit(
        X = training_set[SORTED_FEATURES],
        y = training_set[label_col],
        sample_weight = training_set[sample_weight_col].values,
    )
    return clf

xgb_clf = train_xgb_model(train_df, SORTED_FEATURES)

buffer = BytesIO()
pickle.dump(new_md, buffer)
buffer.seek(0)
s3.put_object(
    Bucket="data-science-bayesian",
    Key="templates/model_training/example_xgb_model.json",
    Body=buffer.getvalue(),
)

In [0]:
def predict_and_plot_xgb(val_df, SORTED_FEATURES, xgb_clf): 
    val_df['prediction'] = xgb_clf.predict_proba(
        val_df[SORTED_FEATURES]
    )[:,1]

    fig, ax = plt.subplots(1,1, figsize = (6,6))
    RocCurveDisplay.from_predictions(val_df['label'], val_df['prediction'], color = 'blue', name = 'example xgb model', ax = ax)
    fig.show()
predict_and_plot_xgb(val_df, SORTED_FEATURES, xgb_clf)